#  Creating enzyme-substrate database from GO Annotation database for Uniprot IDs

### 1. Extracting all GO Terms with information about chemical reactions
### 2. Getting IDs for the substrates
### 3. Searching the GO Annotations database for UniProt IDs for entries with GO Terms stating the catalyitic activity of an enzyme
### 4. Splitting the dataset in training and testing set
### 5. Calculating enzyme and substrate representations
### 6. Adding negative data points
### 7. Adding task-specific enzyme representations (extra token)
### 8. Adding task-specific metabolite representations
### 9. Adding task-specific enzyme representations (mean representations)
### 10. Adding ECFP vectors of different dimensions

In [2]:
import pandas as pd
import numpy as np
import random
from os.path import join
import os
import re
import sys
import time
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from Bio import SeqIO
import warnings
import torch
warnings.filterwarnings("ignore")

sys.path.append('./additional_code')
from data_preprocessing import *

CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)

UnpicklingError: invalid load key, 'v'.

## 1. Extract all GO Terms with information about chemical reactions

### (a) Create pandas DataFrame with alle GO Terms (with infomration about the catalytic activity of enzymes) including its names, definitions, and reaction IDs:
go.obo file with information about GO Terms was downloaded from http://geneontology.org/docs/download-ontology/

In [ ]:
df = pd.DataFrame(columns = ["GO ID", "Definition", "Name", "RHEA ID"])

#read go obo file
path_go_obo = join(CURRENT_DIR, ".." ,"data", "GOA_data", 'go.obo')
file = open(path_go_obo, 'r')
Lines = file.read()

#process data from go.obo-file and save it in a pandas DataFrame:
start = Lines.index('[Term]\n')
while start != -1:
    start = Lines.index('[Term]\n')
    Lines = Lines[start+1:]
    try:
        GO_Term = Lines[: Lines.index('[Term]\n')]
    except ValueError:
        start = -1
    definition, name, ID = get_info_from_GO_Term(GO_Term = GO_Term)
    if "Catalysis of the reaction" in definition:
        RHEA_ID= get_RHEA_reaction_IDs(GO_ID = ID, GO_Term = GO_Term)
        df = df.append({"GO ID" : ID , "Definition" : definition, "Name" : name,
                        "RHEA ID" : RHEA_ID}, ignore_index = True)
df.head()

In [ ]:
df.to_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "df_GO_catalytic.pkl"))

In [ ]:
len(df.loc[~pd.isnull(df["RHEA ID"])])

In [ ]:
df_catalytic = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "df_GO_catalytic.pkl"))
df_catalytic

## 2. Getting IDs for the substrates

### (a) Trying to get substrate name via text mining of the GO Term definitions:

In [ ]:
df_catalytic["substrates"] = ""

for ind in df_catalytic.index:
    df_catalytic["substrates"][ind] = find_substrates(definition = df_catalytic["Definition"][ind])
    
df_catalytic

### (b) Getting substrate CHEBI IDs for the reactions with RHEA IDs:

#### (b)(i) Mapping the metabolites of the reaction with RHEA IDs to CHEBI metabolite IDs
Information about RHEA reactions were downloaded from https://ftp.expasy.org/databases/rhea/txt/

In [ ]:
df_RHEA = pd.DataFrame(columns = ["RHEA ID", "CHEBI IDs", "CHEBI_ID_list"])

file1 = open(join(CURRENT_DIR, ".." ,"data", "reaction_data", "rhea-reactions.txt"), 'r')
Lines = file1.readlines()

while True:
    try:
        end = Lines.index('///\n')
        entry = Lines[:end]
        RHEA_ID, CHEBI_IDs = extract_RHEA_ID_and_CHEBI_IDs(entry)
        CHEBI_ID_list = get_substrate_IDs(IDs = CHEBI_IDs)
        Lines = Lines[end+1:]
        df_RHEA = df_RHEA.append({"RHEA ID" : RHEA_ID, "CHEBI_ID_list" : CHEBI_ID_list}, ignore_index = True)
    except ValueError:
        break
        
df_RHEA["RHEA ID"] = [float(ID.split(":")[-1]) for ID in df_RHEA["RHEA ID"]]
df_RHEA.to_pickle(join(CURRENT_DIR, ".." ,"data", "reaction_data", "RHEA_reaction_df.pkl"))
df_RHEA

#### (b)(ii) Mapping CHEBI IDs to df_catalytic

In [ ]:
df_catalytic["RHEA ID"] = [float(ID) for ID in df_catalytic["RHEA ID"]]
df_catalytic = df_catalytic.merge(df_RHEA, how = "left", on = ["RHEA ID"])
df_catalytic.head()

#### (b)(iii) Creating DataFrame with all combinations of GO Terms and single substrates:

In [ ]:
df_substrates = pd.DataFrame(columns = ["GO ID", "molecule", "molecule ID", "RHEA ID"])
df_catalytic["CHEBI_ID_list"].loc[pd.isnull(df_catalytic["CHEBI_ID_list"])] = ""

for ind in df_catalytic.index:
    GO_ID = df_catalytic["GO ID"][ind]
    RHEA_ID = df_catalytic["RHEA ID"][ind]
    
    if len(df_catalytic["CHEBI_ID_list"][ind]) > 0:
        IDs = df_catalytic["CHEBI_ID_list"][ind]
        for ID in IDs:
            df_substrates = df_substrates.append({"GO ID" : GO_ID, "molecule": np.nan, "molecule ID" : ID,
                                                 "RHEA ID" : RHEA_ID},
                                                 ignore_index = True)
    else:
        metabolites = df_catalytic["substrates"][ind]
        for met in metabolites:
            df_substrates = df_substrates.append({"GO ID" : GO_ID, "molecule": met, "molecule ID" : np.nan,
                                                 "RHEA ID" : RHEA_ID},
                                                 ignore_index = True)
            
df_substrates.reset_index(inplace = True, drop = True)
df_substrates

###  (c) Trying to map all substrates without a CHEBI ID to an identifier:

In [ ]:
metabolites = []
for ind in df_substrates.index:
    if pd.isnull(df_substrates["molecule ID"][ind]):
        metabolites = metabolites + [df_substrates["molecule"][ind]]
    
df_unmapped = pd.DataFrame(data = {"metabolites" : list(set(metabolites))})
df_unmapped

#### (c)(i) Mapping metabolite names to KEGG compound synonym database:

In [ ]:
drugs_df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "KEGG_drugs_df.pkl"))
compounds_df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "KEGG_substrate_df.pkl"))
KEGG_substrate_df = compounds_df.append(drugs_df).reset_index(drop = True)

##If we have multiple IDs for the same substrate name, we keep the first ID:
droplist = []
for ind in KEGG_substrate_df.index:
    if not ind in droplist:
        substrate = KEGG_substrate_df["substrate"][ind]
        help_df = KEGG_substrate_df.loc[KEGG_substrate_df["substrate"] == substrate]
        if len(help_df) > 1 :
            droplist = droplist + list(help_df.index)[1:]

KEGG_substrate_df.drop(droplist, inplace = True)

KEGG_substrate_df["substrate"] = [name.lower() for name in KEGG_substrate_df["substrate"]]
df_unmapped["substrate"] = [name.lower() for name in df_unmapped["metabolites"]]

df_unmapped = df_unmapped.merge(KEGG_substrate_df, on = "substrate", how = "left")
print("For %s out of %s data points we could not map the substrate name to a KEGG ID." %
      (sum(pd.isnull(df_unmapped["KEGG ID"])), len(df_unmapped)))

#### (c)(ii) Mapping all substrate names without a KEGG CID to PubChem CIDs:

In [ ]:
unmapped_substrates = list(set(list(df_unmapped["substrate"][pd.isnull(list(df_unmapped["KEGG ID"]))])))
matches = substrate_names_to_Pubchem_CIDs(unmapped_substrates)
matches.to_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "Pubchem_substrate_matches_GO_Terms.pkl"))

Mapping all PubChem CIDs to KEGG CIDs: Therefore, we create a txt-file ("Pubchem_CIDs_GO_Terms.txt") that contains all PubChem CIDs that we want to map to KEGG CIDs:

In [ ]:
#convert CIDs from strings to integers:
for ind in matches.index:
    cid = matches.loc[ind]["CID"]
    if not pd.isnull(cid):
        matches["CID"][ind] = int(cid)
        
        
#create txt file with all CIDs in matches:
CIDs = list(matches.loc[~ pd.isnull(matches["CID"])]["CID"])
f = open(join(CURRENT_DIR, ".." ,"data", "substrate_data", "Pubchem_CIDs_GO_Terms.txt"),"w") 
for cid in CIDs:
    f.write(str(cid) + "\n")
f.close()

The txt-file "Pubchem_CIDs_GO_Terms.txt" can be used as the input for the webservice https://www.metaboanalyst.ca/MetaboAnalyst/upload/ConvertView.xhtml to map the Pubchem CIDs to KEGG CIDs and CHEBI IDs. 

In [ ]:
matches= pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "Pubchem_substrate_matches_GO_Terms.pkl"))
matches["CID"] = [int(cid) if not pd.isnull(cid) else np.nan for cid in matches["CID"]]
matches.head()
matches = matches.loc[~pd.isnull(matches["CID"])]

#load the resulting file and store it in a DataFrame:
Pubchem_CID_to_KEGG_df = pd.read_csv(join(CURRENT_DIR, ".." ,"data", "substrate_data", "name_map_metaboanalyst.csv"), sep = ",")
#rename columns:
Pubchem_CID_to_KEGG_df.rename(columns = {"PubChem" : "CID"}, inplace = True)
Pubchem_CID_to_KEGG_df.drop(columns = ["Match","Query", "HMDB", "SMILES", "Comment"], inplace = True)

#merge this DataFrame with the DataFrame called matches:
macthes_with_KEGG_IDs = pd.merge(matches, Pubchem_CID_to_KEGG_df, how='left', on=['CID'])
macthes_with_KEGG_IDs.rename(columns = {"Match" : "substrate"}, inplace = True)
macthes_with_KEGG_IDs

In [ ]:
df_unmapped["CHEBI ID"] = np.nan

for ind in df_unmapped.index:
    if pd.isnull(df_unmapped["KEGG ID"][ind]):
        substrate = df_unmapped["substrate"][ind]
        try:
            KEGG_ID = list(macthes_with_KEGG_IDs.loc[macthes_with_KEGG_IDs["substrate"]== substrate]["KEGG ID"])[0]
            df_unmapped["KEGG ID"][ind] = KEGG_ID
        except:
            None
        try:
            CHEBI_ID = list(macthes_with_KEGG_IDs.loc[macthes_with_KEGG_IDs["substrate"]== substrate]["ChEBI"])[0]
            df_unmapped["CHEBI ID"][ind] = CHEBI_ID
        except:
            None
df_unmapped

#### (c)(iii) Mapping substrate IDs to df_substrate:

In [ ]:
for ind in df_substrates.index:
    if pd.isnull(df_substrates["molecule ID"][ind]):
        met = df_substrates["molecule"][ind]

        ID = list(df_unmapped["CHEBI ID"].loc[df_unmapped["metabolites"] == met])[0]
        if pd.isnull(ID):
            ID = list(df_unmapped["KEGG ID"].loc[df_unmapped["metabolites"] == met])[0]

        df_substrates["molecule ID"][ind] = ID
df_substrates

In [ ]:
df_substrates.drop(columns = ["molecule"], inplace= True)
df_substrates = df_substrates.loc[~pd.isnull(df_substrates["molecule ID"])]
df_substrates.reset_index(inplace = True, drop = True)
df_substrates

In [ ]:
df_substrates.to_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "df_GO_catalytic_2.pkl"))

## 3. Searching the GO Annotations database for entries with GO Terms stating the catalyitic activity of an enzyme

In [ ]:
df_substrates = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "df_GO_catalytic_2.pkl"))

In [ ]:
len(df_catalytic) - len(set(df_substrates["GO ID"]))

#### (a) Searching the database

The GO Annotation database contains evidence codes (ECOs). These ECOs contain the information which kind of evidence an annotation has, e.g. "computational", "experimential", "phylogenetic",...
We are only interested in data points with experimental and phylogenetic evidence.

In [ ]:
df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "df_GO_catalytic_2.pkl"))
catalytic_go_terms = list(set(df["GO ID"]))

In [ ]:
len(catalytic_go_terms)

In [ ]:
ECO_to_GAF = pd.read_csv(join(CURRENT_DIR, ".." ,"data", "GOA_data", 'ECO_to_GAF.tsv'), sep = "\t")
exp_evidence = ["EXP","IDA","IPI","IMP","IGI","IEP", "HTP","HDA","HMP","HGI","HEP"]
phylo_evidence = ["IBA","IBD","IKR","IRD"]

The GOA database contains over 922 million entries. We created a function "search_GOA_database" to parallize the search. It takes the argument run which should be an integer betwenn 0 and 922 and searches one million entries accoriding to the value run has.
The GOA database for uniprot IDs was downloaded from here:

In [ ]:
#for run in range(923):
#    search_GOA_database(run)

#### (b)Mapping GO_terms with phylogenetic and experimental evidence to substrate IDs:

Merging the created DataFrames:

In [ ]:
df_GO_UID = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "experimental_and_phylogenetic",
                                "experimental_phylogenetic_df_GO_UID_part_" + str(0) +".pkl"))

for i in range(1,923):
    try:
        df_new = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "experimental_and_phylogenetic",
                                     "experimental_phylogenetic_df_GO_UID_part_" + str(i) +".pkl"))
        df_GO_UID = pd.concat([df_GO_UID, df_new], ignore_index= True)
    except FileNotFoundError:
        print("Error", i)
df_GO_UID.to_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "experimental_phylogenetic_df_GO_UID.pkl"))

In [ ]:
df_GO_UID = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "GOA_data", "experimental_phylogenetic_df_GO_UID.pkl"))

Mapping Uniprot IDs and metabolite IDs:

In [ ]:
df_UID_MID = pd.DataFrame(columns =["Uniprot ID", "molecule ID", "evidence", "RHEA ID"])

for ind in df_GO_UID.index:
    if ind >= -1:
        GO_ID = df_GO_UID["GO Term"][ind]
        try:
            RHEA_ID = list(df_catalytic["RHEA ID"].loc[df_catalytic["GO ID"] == GO_ID])[0]
        except:
            RHEA_ID = np.nan
            print(GO_ID)
        UID = df_GO_UID["Uniprot ID"][ind]
        evidence = df_GO_UID["evidence"][ind]
        met_IDs = list(df_substrates["molecule ID"].loc[df_substrates["GO ID"] == GO_ID])
        for met_ID in met_IDs:
            df_UID_MID = df_UID_MID.append({"Uniprot ID" : UID, "molecule ID" : met_ID,
                                           "evidence": evidence, "RHEA ID" : RHEA_ID}, ignore_index = True)
    if ind % 1000 ==1:
        print(ind)
        
df_UID_MID.drop_duplicates(inplace = True)
df_UID_MID.to_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID.pkl"))

Uniprot_IDs = list(set(df_UID_MID["Uniprot ID"]))
print(len(Uniprot_IDs), len(list(set(df_UID_MID["molecule ID"]))))

df_UID_MID

In [ ]:
df_UID_MID.loc[df_UID_MID["evidence"] == "exp"]

In [ ]:
len(df_UID_MID), len(df_UID_MID.loc[df_UID_MID["evidence"] == "exp"])

We are removing small molecules like H2O and H+ from the dataset:

In [ ]:
'''#magnesium, Phosphoprotein,  Collagen, C02, O2, Calmodulin, water, RNA, calcium, iron, hydrogen,
protein polypeptide chain,DNA, ammonium, lipopolysaccharide, fatty acid anion, double stranded DNA,
iron, hydrogen donor, hydrogen acceptor, lipid''';

remove_mets = ["CHEBI:18420", "C00562", "C00211", "CHEBI:16526", "CHEBI:15379", "C00391", "CHEBI:15377",
               "C00046", "CHEBI:29108", "CHEBI:29034", "CHEBI:15378", "CHEBI:16541", "CHEBI:16991",
              "CHEBI:28938", "CHEBI:16412", "CHEBI:28868", "CHEBI:4705", "CHEBI:29033", "CHEBI:17499",
              "CHEBI:13193", "C01356"]

df_UID_MID = df_UID_MID.loc[~df_UID_MID["molecule ID"].isin(remove_mets)]
df_UID_MID

In [ ]:
len(df_UID_MID), len(df_UID_MID.loc[df_UID_MID["evidence"] == "exp"])

In [ ]:
29603-23384

In [ ]:
len(df_UID_MID.loc[df_UID_MID["evidence"] == "exp"].loc[~pd.isnull(df_UID_MID["RHEA ID"])])

In [ ]:
df_UID_MID_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data", "df_UID_MID_train.pkl" ))
df_UID_MID_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID_test.pkl"))

In [ ]:
df_UID_MID2 = df_UID_MID.loc[df_UID_MID["evidence"] == "exp"]
df = df_UID_MID_train.loc[df_UID_MID_train["evidence"] == "exp"]
df.drop(columns = "evidence", inplace = True)
df_train = df.merge(df_UID_MID2, on = ["Uniprot ID", "molecule ID"], how = "left")

df = df_UID_MID_test.loc[df_UID_MID_test["evidence"] == "exp"]
df.drop(columns = "evidence", inplace = True)
df_test = df.merge(df_UID_MID2, on = ["Uniprot ID", "molecule ID"], how = "left")

len(df_train.loc[~pd.isnull(df_train["RHEA ID"])]) +len(df_test.loc[~pd.isnull(df_test["RHEA ID"])])

In [ ]:
df_UID_MID2 = df_UID_MID.loc[df_UID_MID["evidence"] == "phylo"]
df = df_UID_MID_train.loc[df_UID_MID_train["evidence"] == "phylo"]
df.drop(columns = "evidence", inplace = True)
df_train = df.merge(df_UID_MID2, on = ["Uniprot ID", "molecule ID"], how = "left")

df = df_UID_MID_test.loc[df_UID_MID_test["evidence"] == "phylo"]
df.drop(columns = "evidence", inplace = True)
df_test = df.merge(df_UID_MID2, on = ["Uniprot ID", "molecule ID"], how = "left")

len(df_train.loc[~pd.isnull(df_train["RHEA ID"])]) +len(df_test.loc[~pd.isnull(df_test["RHEA ID"])])

#### (c) Getting an overview of the number of unique enzymes and substrates in the dataset:

In [ ]:
df_exp = df_UID_MID.loc[df_UID_MID["evidence"] == "exp"]
df_phylo = df_UID_MID.loc[df_UID_MID["evidence"] == "phylo"]

print("We have %s entries with phylogenetic evidence and %s entries with experimental evidence" % (len(df_phylo), len(df_exp)))

print("\n experimental dataset:")
print("Number of different enzymes: %s, Number of different substrates: %s"
      % (len(set(df_exp["Uniprot ID"])), len(set(df_exp["molecule ID"]))) )

print("\n phylogenetic dataset:")
print("Number of different enzymes: %s, Number of different substrates: %s"
      % (len(set(df_phylo["Uniprot ID"])), len(set(df_phylo["molecule ID"]))) )

In [ ]:
df_UID_MID.to_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID.pkl"))

## 4. Splitting the dataset in training and testing set:

In [ ]:
df_UID_MID = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID.pkl"))

To split the dataset in training and test set, we take the enzyme's sequence identity into account. Therefore, we first cluster the enzymes by sequence identity:

### (a) Downloading sequences for all UniProt IDs:

Creating file with all UniProt IDs: We split it into two files, since UniProt only takes files up to 2MB.

In [ ]:
Uniprot_IDs = list(set(df_UID_MID["Uniprot ID"]))

Uniprot_IDs_V1 = Uniprot_IDs[:100000]
Uniprot_IDs_V2 = Uniprot_IDs[100000:200000]
Uniprot_IDs_V3 = Uniprot_IDs[200000:]

f = open(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "UNIPROT_IDs_V1.txt"),"w")
for ID in list(set(Uniprot_IDs_V1)):
    f.write(str(ID) + "\n")
f.close()

f = open(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "UNIPROT_IDs_V2.txt"),"w")
for ID in list(set(Uniprot_IDs_V2)):
    f.write(str(ID) + "\n")
f.close()

f = open(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "UNIPROT_IDs_V3.txt"),"w")
for ID in list(set(Uniprot_IDs_V3)):
    f.write(str(ID) + "\n")
f.close()

Downloading UniProt IDs via the UniProt mapping service (https://www.uniprot.org/uploadlists/):

In [ ]:
from Bio import SeqIO

In [ ]:
UNIPROT_df = pd.DataFrame(columns = ["Uniprot ID", "Sequence"])

for version in ["1", "2", "3"]:
    fasta_sequences = SeqIO.parse(open(join(CURRENT_DIR, ".." ,"data", "enzyme_data",
                                            "UNIPROT_IDs_V" +version + ".fasta")),'fasta')
    for fasta in fasta_sequences:
        name, sequence = fasta.id, str(fasta.seq)
        UNIPROT_df = UNIPROT_df.append({"Uniprot ID" : name.split("|")[1], "Sequence" : sequence}, ignore_index = True)

In [ ]:
UNIPROT_df

In [ ]:
UNIPROT_df.to_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "UNIPROT_df.pkl"))

### (b) Clustering enzymes by sequence identity:

In [ ]:
UNIPROT_df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "UNIPROT_df.pkl"))

Creating fasta file with all sequences:

In [ ]:
ofile = open(join(CURRENT_DIR, ".." ,"data", "enzyme_data", 'clusters', "all_sequences.fasta"), "w")
for ind in UNIPROT_df.index:
    seq = UNIPROT_df["Sequence"][ind]
    if not pd.isnull(seq):
        seq_end = seq.find("#")
        seq = seq[:seq_end]
        ofile.write(">" + str(ind) + "\n" + seq  + "\n")
ofile.close()

#### (b)(i) Clustering:

We first cluster in such a way that two different clusters do not contain two enzymes with a sequence identity higher than 80%:

Getting input for cd-hit algorithm:

In [ ]:
# cluster the fasta files
cluster_folder = join(CURRENT_DIR, ".." ,"data", "enzyme_data", 'clusters')
start_folder = cluster_folder
cluster_all_levels(start_folder, 
                   cluster_folder, 
                   filename='all_sequences')

In [ ]:
cluster_folder = join(CURRENT_DIR, ".." ,"data", "enzyme_data", 'clusters')
    
# cluster the fasta files
start_folder = cluster_folder
cluster_all_levels_80(start_folder, 
                   cluster_folder, 
                   filename='all_sequences')

# collect cluster members
df_80 = find_cluster_members_80(folder=cluster_folder, 
                          filename='all_sequences')

display(df_80.describe())
display(df_80.head())
display(df_80.tail())

We cluster in such a way that two different clusters do not contain two enzymes with a sequence identity higher than 60%:

In [ ]:
cluster_all_levels_60(start_folder, 
                   cluster_folder, 
                   filename='all_sequences')

# collect cluster members
df_60 = find_cluster_members_60(folder=cluster_folder, 
                       filename='all_sequences')
display(df_60.describe())

We first cluster in such a way that two different clusters do not contain two enzymes with a sequence identity higher than 40%:

In [ ]:
# cluster the fasta files
cluster_all_levels(start_folder, 
                   cluster_folder, 
                   filename='all_sequences')

# collect cluster members
df_40 = find_cluster_members(folder=cluster_folder, 
                          filename='all_sequences')

display(df_40.describe())

#### (a)(ii) Splitting the dataset in train, validation and test set with a sequence identity cutoff of 80%. Later, we divide the test set in three subparts with identity cutoffs of <40%, 40-60% and 60-80%

In [ ]:
UNIPROT_df["cluster"] = np.nan
for ind in df_80.index:
    member = int(df_80["member"][ind])
    cluster = df_80["cluster"][ind]
    UNIPROT_df["cluster"][member] = cluster

We only use the additional validation set for the training of the ESM-1b model. It is so big that we can't do a 5-fold CV. Therefore, we use an independent validation set to validate the model instead.

In [ ]:
clusters = list(set(UNIPROT_df["cluster"]))
random.seed(1)
random.shuffle(clusters)
print(len(clusters))

n = int(len(clusters)*0.8)
train_clusters = clusters[:n]
test_clusters = clusters[n:]

training_UIDs = UNIPROT_df["Uniprot ID"].loc[UNIPROT_df["cluster"].isin(train_clusters)]
test_UIDs = UNIPROT_df["Uniprot ID"].loc[UNIPROT_df["cluster"].isin(test_clusters)]

df_80["split"] = np.nan
df_80["split"].loc[df_80["cluster"].isin(train_clusters)] = "train"
df_80["split"].loc[df_80["cluster"].isin(test_clusters)] = "test"

train_members = list(df_80["member"].loc[df_80["split"] == "train"])
test_members = list(df_80["member"].loc[df_80["split"] == "test"])

df_60["split"] = np.nan
df_40["split"] = np.nan
df_60["split"].loc[df_60["member"].isin(train_members)] = "train"
df_60["split"].loc[df_60["member"].isin(test_members)] = "test"
df_40["split"].loc[df_40["member"].isin(train_members)] = "train"
df_40["split"].loc[df_40["member"].isin(test_members)] = "test"

len(training_UIDs), len(test_UIDs)

In [ ]:
df_UID_MID_train = df_UID_MID.loc[df_UID_MID["Uniprot ID"].isin(training_UIDs)]
df_UID_MID_test = df_UID_MID.loc[df_UID_MID["Uniprot ID"].isin(test_UIDs)]
len(df_UID_MID_test), len(df_UID_MID_train)

Calculating for every sequence in the validation and test set the maximum accuracy compared to sequences in the training set:

In [ ]:
df_80["identity"] = np.nan
df_80["identity"].loc[df_80["split"].isin(["test"])] =  "60-80%"

test_indices = list(df_80.loc[~pd.isnull(df_80["identity"])].index)


for ind in test_indices:

    member = df_80["member"][ind]
    cluster = list(df_40["cluster"].loc[df_40["member"] == member])[0]
    cluster_splits = list(df_40["split"].loc[df_40["cluster"] == cluster])
    if not "train" in cluster_splits:
        df_80["identity"][ind] = "<40%"
    else:
        cluster = list(df_60["cluster"].loc[df_60["member"] == member])[0]
        cluster_splits = list(df_60["split"].loc[df_60["cluster"] == cluster])
        if not "train" in cluster_splits:
            df_80["identity"][ind] = "40-60%"
            
    if ind % 1000 == 0:
        print(ind)
                    
                    
ind = 0
UNIPROT_df["identity"] = np.nan
for ind in UNIPROT_df.index:
    try:
        UNIPROT_df["identity"][ind] = list(df_80["identity"].loc[df_80["member"] == str(ind)])[0]
    except:
        None
        
UNIPROT_df.to_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data",
                          "Uniprot_df_with_seq_identities.pkl"))

In [ ]:
len(UNIPROT_df.loc[UNIPROT_df["identity"] == "60-80%"]), len(UNIPROT_df.loc[UNIPROT_df["identity"] == "40-60%"]), len(UNIPROT_df.loc[UNIPROT_df["identity"] == "<40%"])

## 5. Calculating enzyme and substrate representations

In [ ]:
UNIPROT_df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "Uniprot_df_with_seq_identities.pkl"))
df_UID_MID = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID.pkl"))

### (a) Calculate extended-connectivity fingerprints (ECFPs) for substrates:

In [ ]:
mol_folder = "C:\\Users\\alexk\\mol-files\\"
mol_IDs = list(set(df_UID_MID["molecule ID"]))

df_ecfps = pd.DataFrame(data = {"substrate ID" : mol_IDs})
df_ecfps["ECFP"] = ""
df_ecfps

Creating txt file with all ChEBI IDs:

In [ ]:
ChEBI_IDs = list(set(df_UID_MID["molecule ID"]))
f = open(join(CURRENT_DIR, ".." ,"data", "substrate_data", "ChEBI_IDs.txt"),"w")
for ID in list(set(ChEBI_IDs)):
    if ID[:2] == "CH":
        f.write(str(ID.replace("CHEBI", "ChEBI")) + "\n")
f.close()

First we map ChEBI IDs to InChI strings using https://metacyc.org/metabolite-translation-service.shtml:

In [ ]:
df_chebi_to_inchi = pd.read_csv(join(CURRENT_DIR, ".." ,"data", "substrate_data", "chebiID_to_inchi.tsv"), sep = "\t")
df_chebi_to_inchi.head()

Mapping InChI string to DataFrame:

In [ ]:
for ind in df_ecfps.index:
    met_ID = df_ecfps["substrate ID"][ind]
    is_CHEBI_ID = (met_ID[0:5] == "CHEBI")
    
    
    if is_CHEBI_ID:
        try:
            ID = int(met_ID.split(" ")[0].split(":")[-1])
            Inchi = list(df_chebi_to_inchi["Inchi"].loc[df_chebi_to_inchi["ChEBI"] == float(ID)])[0]
            if not pd.isnull(Inchi):
                mol = Chem.inchi.MolFromInchi(Inchi)
        except IndexError:
            mol = None
        
    else:
        try:
            mol = Chem.MolFromMolFile(mol_folder +  "/mol-files/" + met_ID + '.mol')
        except OSError:
            None
            
    if mol is not None:
        ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=1024).ToBitString()
        df_ecfps["ECFP"][ind] = ecfp

In [ ]:
df_ecfps = df_ecfps.loc[df_ecfps["ECFP"] !=""]

In [ ]:
df_ecfps.to_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "df_ecfps.pkl"))

### (b) Calculating enzyme representations with the ESM-1b model:

The ESM-1b model takes a FASTA file with the enzymes' amino acid sequences as its input. Creating a FASTA file with enzyme sequences:

In [ ]:
ofile = open(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "all_sequences.fasta"), "w")
for ind in UNIPROT_df.index:
    seq = UNIPROT_df["Sequence"][ind]
    if not pd.isnull(seq):
        seq_end = seq.find("#")
        seq = seq[:seq_end]
        ofile.write(">" + str(ind) + "\n" + seq[:1018]  + "\n")
ofile.close()

To calculate the ESM-1b vectors, we used the model and code provided by the Facebook Research team: https://github.com/facebookresearch/esm. The following command line was used to calculate the representations:

<span style="color:red"> 
python extract.py esm1b_t33_650M_UR50S \path_to_fasta_file\all_sequences.fasta \path_to_store_representations\all_sequences_esm1b_reps--repr_layers 33 --include mean </span>

Loading ESM-1b-vectors and storing them in the UniProt DataFrame. All representations were merged into one dictionary:

In [ ]:
UNIPROT_df["ESM1b"] = ""

rep_dict = torch.load(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "all_sequences_enz_sub.pt"))

for ind in UNIPROT_df.index:
    try:
        UNIPROT_df["ESM1b"][ind] = rep_dict[str(ind) +".pt"]
    except:
        print(ind)
UNIPROT_df

In [ ]:
UNIPROT_df.to_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "Uniprot_df_with_ESM1b.pkl"))

## 6. Adding negative data points

In [ ]:
UNIPROT_df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "Uniprot_df.pkl"))
df_UID_MID = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID.pkl"))

In [ ]:
training_UIDs, testing_UIDs = list(set(df_UID_MID_train["Uniprot ID"])), list(set(df_UID_MID_test["Uniprot ID"]))

In [ ]:
df_UID_MID_train = df_UID_MID.loc[df_UID_MID["Uniprot ID"].isin(training_UIDs)]
df_UID_MID_test = df_UID_MID.loc[df_UID_MID["Uniprot ID"].isin(testing_UIDs)]

Only using UniProt IDs with ESM1b vectors:

In [ ]:
#UIDs_with_rep = list(set(UNIPROT_df["Uniprot ID"].loc[UNIPROT_df["ESM1b"] != ""]))

#df_UID_MID_train = df_UID_MID_train.loc[df_UID_MID_train["Uniprot ID"].isin(UIDs_with_rep)]
#df_UID_MID_test = df_UID_MID_test.loc[df_UID_MID_test["Uniprot ID"].isin(UIDs_with_rep)]

df_UID_MID_train = df_UID_MID_train.sample(frac = 1, random_state = 1).reset_index(drop = True)
df_UID_MID_test = df_UID_MID_test.sample(frac = 1, random_state = 1).reset_index(drop = True)

Mapping ECFPs to DataFrames:

In [ ]:
df_ecfps = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "df_ecfps.pkl"))

df_UID_MID_train["ECFP"] = ""
df_UID_MID_test["ECFP"] = ""
for ind in df_ecfps.index:
    sub_ID, ecfp = df_ecfps["substrate ID"][ind], df_ecfps["ECFP"][ind]
    df_UID_MID_train["ECFP"].loc[df_UID_MID_train["molecule ID"] == sub_ID] = ecfp
    df_UID_MID_test["ECFP"].loc[df_UID_MID_test["molecule ID"] == sub_ID] = ecfp
    
df_UID_MID_test = df_UID_MID_test.loc[df_UID_MID_test["ECFP"] != ""]
df_UID_MID_train = df_UID_MID_train.loc[df_UID_MID_train["ECFP"] != ""]

#df_UID_MID_train.to_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data", "df_UID_MID_train.pkl"))
#df_UID_MID_test.to_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID_test.pkl"))

In [ ]:
df_all = pd.concat([df_UID_MID_train, df_UID_MID_test], ignore_index = True)

df_exp = df_all.loc[df_all["evidence"] == "exp"]
df_phylo = df_all.loc[df_all["evidence"] == "phylo"]

print("We have %s entries with phylogenetic evidence and %s entries with experimental evidence" % (len(df_phylo), len(df_exp)))

print("\n experimental dataset:")
print("Number of different enzymes: %s, Number of different substrates: %s"
      % (len(set(df_exp["Uniprot ID"])), len(set(df_exp["molecule ID"]))) )

print("\n phylogenetic dataset:")
print("Number of different enzymes: %s, Number of different substrates: %s"
      % (len(set(df_phylo["Uniprot ID"])), len(set(df_phylo["molecule ID"]))))

### (a) Creating negative data points for the training and test set:
To assign the negative data points, we will choose similar metabolites compared to the substrate of the psoitive datapoints. Therefore, we are first creating a similarity matrix for all metabolites in the dataset.

In [ ]:
df_chebi_to_inchi = pd.read_csv(join(CURRENT_DIR, ".." ,"data", "substrate_data", "chebiID_to_inchi.tsv"), sep = "\t")
mol_folder = "C:\\Users\\alexk\\mol-files\\"

count = 0

def get_mol(met_ID):
    is_CHEBI_ID = (met_ID[0:5] == "CHEBI")
    is_InChI = (met_ID[0:5] == "InChI")
    if is_CHEBI_ID:
        try:
            ID = int(met_ID.split(" ")[0].split(":")[-1])
            Inchi = list(df_chebi_to_inchi["Inchi"].loc[df_chebi_to_inchi["ChEBI"] == float(ID)])[0]
            mol = Chem.inchi.MolFromInchi(Inchi)
        except:
            mol = None     
    elif is_InChI:
        try:
            mol = Chem.inchi.MolFromInchi(met_ID)
        except:
            mol = None
        
    else:
        try:
            mol = Chem.MolFromMolFile(mol_folder +  "mol-files\\" + met_ID + '.mol')
        except OSError:
            mol = None
            
    return(mol)

def drop_samples_without_mol_file(df):
    droplist = []
    for ind in df.index:
        if get_mol(met_ID = df["molecule ID"][ind]) is None:
            droplist.append(ind)

    df.drop(droplist, inplace = True)
    return(df)

def get_metabolites_and_similarities(df):
    df_metabolites = pd.DataFrame(data = {"ECFP": df["ECFP"], "ID": df["molecule ID"]})
    df_metabolites = df_metabolites.drop_duplicates()
    df_metabolites.reset_index(inplace = True, drop = True)


    ms = [get_mol(met_ID = df_metabolites["ID"][ind]) for ind in df_metabolites.index]
    fps = [Chem.RDKFingerprint(x) for x in ms]

    similarity_matrix = np.zeros((len(ms), len(ms)))
    for i in range(len(ms)):
        for j in range(len(ms)):
            similarity_matrix[i,j] = DataStructs.FingerprintSimilarity(fps[i],fps[j])
            
    return(df_metabolites, similarity_matrix)



def get_valid_list(met_ID, UID, forbidden_metabolites, df_metabolites, similarity_matrix, lower_bound =0.7, upper_bound =0.9):
    binding_met_IDs = list(df_UID_MID["molecule ID"].loc[df_UID_MID["Uniprot ID"] == UID])
    k = df_metabolites.loc[df_metabolites["ID"] == met_ID].index[0]

    similarities = similarity_matrix[k,:]
    selection = (similarities< upper_bound) * (similarities >lower_bound) 
    metabolites = list(df_metabolites["ID"].loc[selection])
    
    no_mets = list(set(binding_met_IDs + forbidden_metabolites))
    
    metabolites = [met for met in metabolites if (met not in no_mets)]
    return(metabolites)


def create_negative_samples(df, df_metabolites, similarity_matrix):
    start = time.time()
    UID_list = []
    MID_list = []
    Type_list = []
    forbidden_mets = []

    for ind in df.index:
        if ind % 100 ==0:
            print(ind)
            print("Time: %s [min]" % np.round(float((time.time()-start)/60),2))

            df2 = pd.DataFrame(data = {"Uniprot ID": UID_list, "molecule ID" : MID_list, "type" : Type_list})
            df2["Binding"] = 0
            df = pd.concat([df, df2], ignore_index=True)

            UID_list, MID_list, Type_list = [], [], []

            forbidden_mets_old = forbidden_mets.copy()
            all_mets = list(set(df["molecule ID"]))
            all_mets = [met for met in all_mets if not met in forbidden_mets_old]
            forbidden_mets = list(set([met for met in all_mets if 
                                       (np.mean(df["Binding"].loc[df["molecule ID"] == met]) < 1/4)]))
            forbidden_mets = forbidden_mets + forbidden_mets_old
            print(len(forbidden_mets))

        UID = df["Uniprot ID"][ind]
        Type = df["type"][ind]
        met_ID = df["molecule ID"][ind]

        metabolites = get_valid_list(met_ID = met_ID, UID = UID, forbidden_metabolites= forbidden_mets,
                                     df_metabolites = df_metabolites, similarity_matrix = similarity_matrix,
                                     lower_bound =0.7, upper_bound =0.95)
        lower_bound = 0.7
        while len(metabolites) < 2:
            lower_bound = lower_bound - 0.2
            metabolites = get_valid_list(met_ID = met_ID, UID = UID, forbidden_metabolites= forbidden_mets,
                                     df_metabolites = df_metabolites, similarity_matrix = similarity_matrix,
                                     lower_bound =lower_bound, upper_bound =0.95)
            if lower_bound <0:
                break
        
        if lower_bound < 0.7:
            global count
            count +=1
        
        new_metabolites =  random.sample(metabolites, min(3,len(metabolites)))

        for met in new_metabolites:
            UID_list.append(UID), MID_list.append(met), Type_list.append(Type)

    df2 = pd.DataFrame(data = {"Uniprot ID": UID_list, "molecule ID" : MID_list, "type" : Type_list})
    df2["Binding"] = 0

    df = pd.concat([df, df2], ignore_index = True)
    return(df)

#### (a)(i) Creating negative data points for the training set (experimental evidence):

In [ ]:
df_UID_MID_train_exp = df_UID_MID_train.loc[df_UID_MID_train["evidence"] == "exp"]

df_UID_MID_train_exp = drop_samples_without_mol_file(df = df_UID_MID_train_exp)
#calculating similarity matrix for all metabolites in the set:
df_metabolites_train, similarity_matrix_train = get_metabolites_and_similarities(df = df_UID_MID_train_exp)
print(len(df_metabolites_train))

df_UID_MID_train_exp["Binding"] = 1
df_UID_MID_train_exp.reset_index(inplace = True, drop = True)

df_UID_MID_train_exp = create_negative_samples(df = df_UID_MID_train_exp, df_metabolites = df_metabolites_train,
                                          similarity_matrix = similarity_matrix_train)
df_UID_MID_train_exp

In [ ]:
(count)/len(df_UID_MID_train_exp.loc[df_UID_MID_train_exp["Binding"] == 0])

#### (a)(ii) Creating negative data points for the training set (phylogentical evidence):

In [ ]:
df_UID_MID_train_phylo = df_UID_MID_train.loc[df_UID_MID_train["evidence"] == "phylo"]

In [ ]:
df_UID_MID_train_phylo = drop_samples_without_mol_file(df = df_UID_MID_train_phylo)
#calculating similarity matrix for all metabolites in the set:
df_metabolites_train, similarity_matrix_train = get_metabolites_and_similarities(df = df_UID_MID_train_phylo)
print(len(df_metabolites_train))

df_UID_MID_train_phylo["Binding"] = 1
df_UID_MID_train_phylo.reset_index(inplace = True, drop = True)

df_UID_MID_train_phylo = create_negative_samples(df = df_UID_MID_train_phylo, df_metabolites = df_metabolites_train,
                                          similarity_matrix = similarity_matrix_train)
df_UID_MID_train_phylo

461
116600
Time: 166.94 [min]
461
116700
Time: 167.07 [min]
461
116800
Time: 167.21 [min]
461
116900
Time: 167.35 [min]
461
117000
Time: 167.48 [min]
461
117100
Time: 167.62 [min]
461
117200
Time: 167.75 [min]
461
117300
Time: 167.9 [min]
462
117400
Time: 168.04 [min]
463
117500
Time: 168.17 [min]
464
117600
Time: 168.3 [min]
464
117700
Time: 168.44 [min]
464
117800
Time: 168.58 [min]
464
117900
Time: 168.71 [min]
464
118000
Time: 168.85 [min]
464
118100
Time: 168.98 [min]
465
118200
Time: 169.12 [min]
465
118300
Time: 169.25 [min]
466
118400
Time: 169.39 [min]
466
118500
Time: 169.52 [min]
466
118600
Time: 169.65 [min]
467
118700
Time: 169.8 [min]
467
118800
Time: 169.94 [min]
467
118900
Time: 170.08 [min]
467
119000
Time: 170.22 [min]
468
119100
Time: 170.36 [min]
468
119200
Time: 170.51 [min]
469
119300
Time: 170.65 [min]
469
119400
Time: 170.79 [min]
469
119500
Time: 170.93 [min]
470
119600
Time: 171.07 [min]
471
119700
Time: 171.21 [min]
472
119800
Time: 171.35 [min]
473
119900
Ti

144000
Time: 202.85 [min]
534
144100
Time: 202.98 [min]
534
144200
Time: 203.1 [min]
534
144300
Time: 203.23 [min]
534
144400
Time: 203.35 [min]
534
144500
Time: 203.47 [min]
534
144600
Time: 203.59 [min]
535
144700
Time: 203.71 [min]
535
144800
Time: 203.83 [min]
535
144900
Time: 203.95 [min]
535
145000
Time: 204.07 [min]
535
145100
Time: 204.18 [min]
535
145200
Time: 204.31 [min]
535
145300
Time: 204.42 [min]
535
145400
Time: 204.55 [min]
535
145500
Time: 204.67 [min]
536
145600
Time: 204.79 [min]
536
145700
Time: 204.91 [min]
536
145800
Time: 205.02 [min]
537
145900
Time: 205.15 [min]
537
146000
Time: 205.26 [min]
538
146100
Time: 205.39 [min]
539
146200
Time: 205.52 [min]
539
146300
Time: 205.64 [min]
539
146400
Time: 205.76 [min]
539
146500
Time: 205.89 [min]
539
146600
Time: 206.02 [min]
540
146700
Time: 206.14 [min]
541
146800
Time: 206.27 [min]
541
146900
Time: 206.4 [min]
541
147000
Time: 206.52 [min]
541
147100
Time: 206.65 [min]
541
147200
Time: 206.77 [min]
541
147300
Time:

171400
Time: 234.53 [min]
606
171500
Time: 234.65 [min]
606
171600
Time: 234.76 [min]
606
171700
Time: 234.87 [min]
606
171800
Time: 234.98 [min]
608
171900
Time: 235.08 [min]
610
172000
Time: 235.2 [min]
610
172100
Time: 235.31 [min]
611
172200
Time: 235.42 [min]
611
172300
Time: 235.53 [min]
611
172400
Time: 235.65 [min]
611
172500
Time: 235.76 [min]
611
172600
Time: 235.87 [min]
611
172700
Time: 235.97 [min]
612
172800
Time: 236.08 [min]
612
172900
Time: 236.19 [min]
612
173000
Time: 236.3 [min]
613
173100
Time: 236.42 [min]
613
173200
Time: 236.53 [min]
613
173300
Time: 236.64 [min]
613
173400
Time: 236.75 [min]
613
173500
Time: 236.87 [min]
613
173600
Time: 236.97 [min]
613
173700
Time: 237.08 [min]
613
173800
Time: 237.19 [min]
613
173900
Time: 237.3 [min]
613
174000
Time: 237.41 [min]
613
174100
Time: 237.52 [min]
613
174200
Time: 237.63 [min]
613
174300
Time: 237.74 [min]
614
174400
Time: 237.85 [min]
614
174500
Time: 237.96 [min]
614
174600
Time: 238.07 [min]
614
174700
Time: 

198800
Time: 263.37 [min]
649
198900
Time: 263.48 [min]
649
199000
Time: 263.59 [min]
650
199100
Time: 263.69 [min]
650
199200
Time: 263.79 [min]
650
199300
Time: 263.88 [min]
650
199400
Time: 263.98 [min]
650
199500
Time: 264.08 [min]
650
199600
Time: 264.19 [min]
651
199700
Time: 264.29 [min]
651
199800
Time: 264.39 [min]
651
199900
Time: 264.48 [min]
651
200000
Time: 264.58 [min]
651
200100
Time: 264.69 [min]
651
200200
Time: 264.79 [min]
651
200300
Time: 264.89 [min]
651
200400
Time: 265.0 [min]
652
200500
Time: 265.1 [min]
652
200600
Time: 265.19 [min]
652
200700
Time: 265.29 [min]
652
200800
Time: 265.4 [min]
652
200900
Time: 265.49 [min]
652
201000
Time: 265.6 [min]
652
201100
Time: 265.69 [min]
653
201200
Time: 265.8 [min]
653
201300
Time: 265.89 [min]
653
201400
Time: 266.0 [min]
653
201500
Time: 266.1 [min]
653
201600
Time: 266.19 [min]
653
201700
Time: 266.29 [min]
653
201800
Time: 266.37 [min]
653
201900
Time: 266.47 [min]
653
202000
Time: 266.57 [min]
653
202100
Time: 266.

,Uniprot ID,molecule ID,evidence,ECFP,Binding,type
0,A8XT89,CHEBI:58885,phylo,0000000000000100000000000000000000000000000000...,1,NaN
1,B2GV06,CHEBI:57292,phylo,0100100001000000000000000000000011000000000000...,1,NaN
2,A0A022RBJ3,CHEBI:33227,phylo,1000000000000000000000000000000000000000000000...,1,NaN
3,G3S168,CHEBI:59776,phylo,0100000000000000000000000000000000000000000000...,1,NaN
4,F6I0H0,C00002,phylo,0000000001000000000000000000000000000000000000...,1,NaN
...,...,...,...,...,...,...
794423,A0A2J6JMI2,CHEBI:30616,NaN,NaN,0,NaN
794424,Q8Y7G6,C00002,NaN,NaN,0,NaN
794425,Q8Y7G6,CHEBI:30616,NaN,NaN,0,NaN
794426,B9R6X0,C00002,NaN,NaN,0,NaN


#### (a)(iii) Creating negative data points for the test set:

In [ ]:
df_UID_MID_test = df_UID_MID_test.loc[df_UID_MID_test["evidence"] == "exp"]

In [ ]:
df_UID_MID_test = drop_samples_without_mol_file(df = df_UID_MID_test)
#calculating similarity matrix for all metabolites in the set:
df_metabolites_test, similarity_matrix_test = get_metabolites_and_similarities(df = df_UID_MID_test)
print(len(df_metabolites_test))

df_UID_MID_test["Binding"] = 1
df_UID_MID_test.reset_index(inplace = True, drop = True)

df_UID_MID_test = create_negative_samples(df = df_UID_MID_test, df_metabolites = df_metabolites_test,
                                          similarity_matrix = similarity_matrix_test)
df_UID_MID_test

In [ ]:
df_UID_MID_train_phylo.to_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data","df_UID_MID_train_phylo.pkl"))
df_UID_MID_train_exp.to_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data","df_UID_MID_train_exp.pkl"))
df_UID_MID_test.o_picktle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data","df_UID_MID_test_exp_phylo.pkl"))

### (b) Mapping ECFPs and ESM-1b-vectors to different splits:

In [ ]:
df_UID_MID_train_phylo = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data","df_UID_MID_train_phylo.pkl"))
df_UID_MID_train_exp = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data","df_UID_MID_train_exp.pkl"))
df_UID_MID_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data","df_UID_MID_test_exp_phylo.pkl"))

In [ ]:
df_UID_MID_train_exp["evidence"] = "exp"
df_UID_MID_train_phylo["evidence"] = "phylo"
df_UID_MID_train = pd.concat([df_UID_MID_train_exp, df_UID_MID_train_phylo], ignore_index = True)

df_UID_MID_test["evidence"] = "exp"

#### (b)(i) Mappings ECFPs:

In [ ]:
df_UID_MID_train.drop(columns = ["ECFP"], inplace = True)
df_UID_MID_train = df_UID_MID_train.merge(df_ecfps, right_on = "substrate ID", left_on = "molecule ID", how = "left")

df_UID_MID_test.drop(columns = ["ECFP"], inplace = True)
df_UID_MID_test = df_UID_MID_test.merge(df_ecfps, right_on = "substrate ID", left_on = "molecule ID", how = "left")

#### (b)(ii) Mappings ESM1b-vectors:

In [ ]:
df_train = df_UID_MID_train
df_test = df_UID_MID_test

Uniprot_df = pd.DataFrame(data = {"Uniprot ID" : UNIPROT_df["Uniprot ID"],
                                 "ESM1b" : UNIPROT_df["ESM1b"]})

df_train = df_train.merge(Uniprot_df, on = "Uniprot ID", how = "left")
df_test = df_test.merge(Uniprot_df, on = "Uniprot ID", how = "left")

df_train.to_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train.pkl"))
df_test.to_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test.pkl"))

## 7. Adding task-specific enzyme representations (extra token)

### (a) Creating input data for training the task-specific ESM1b vectors:

In [ ]:
df_UID_MID_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","enzyme_substrate_data", "df_UID_MID_train.pkl" ))
df_UID_MID_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_substrate_data", "df_UID_MID_test.pkl"))

In [ ]:
df_UID_MID_train.loc[df_UID_MID_train["evidence"] == "exp"]

### (b) Training the ESM1b model:
Training is executed via terminal with the following command:

### (c) Creating and mapping task-specific ESM1b vectors:

The following command line was used to calculate the representations:

<span style="color:red"> 
python extract_ts.py esm1b_t33_650M_UR50S \path_to_fasta_file\all_sequences.fasta \path_to_store_representations\all_sequences_enz_sub_ts --repr_layers 33 --include mean </span>

Loading ESM-1b-vectors and storing them in the UniProt DataFrame. All representations were merged into one dictionary:

In [ ]:
UNIPROT_df = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "UNIPROT_df.pkl"))
ofile = open(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "all_sequences.fasta"), "w")
for ind in UNIPROT_df.index:
    seq = UNIPROT_df["Sequence"][ind]
    if not pd.isnull(seq):
        seq_end = seq.find("#")
        seq = seq[:seq_end]
        ofile.write(">" + str(ind) + "\n" + seq[:1018]  + "\n")
ofile.close()

In [ ]:
rep_dict = torch.load(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "all_sequences_enz_sub_ts.pt"))

UNIPROT_df["ESM1b_ts"] = ""

for ind in UNIPROT_df.index:
    try:
        UNIPROT_df["ESM1b_ts"][ind] = rep_dict[str(ind) +".pt"]
    except:
        print(ind)
UNIPROT_df

Mapping:

In [ ]:
df_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train.pkl"))
df_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test.pkl"))

Uniprot_df = pd.DataFrame(data = {"Uniprot ID" : UNIPROT_df["Uniprot ID"],
                                 "ESM1b_ts" : UNIPROT_df["ESM1b_ts"]})

df_train = df_train.merge(Uniprot_df, on = "Uniprot ID", how = "left")
df_test = df_test.merge(Uniprot_df, on = "Uniprot ID", how = "left")

df_train.to_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train_with_ESM1b_ts.pkl"))
df_test.to_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test_with_ESM1b_ts.pkl"))

## 8. Adding task-specific metabolite representations

Task-specific metabolite representations are created with the jupyter notebook "Training Graph Neural Network":

In [ ]:
df_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train_with_ESM1b_ts_GNN.pkl"))
df_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test_with_ESM1b_ts_GNN.pkl"))

## 9. Adding task-specific enzyme representations (mean representations)

In [ ]:
df_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train_with_ESM1b_ts_GNN.pkl"))
df_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test_with_ESM1b_ts_GNN.pkl"))

rep_dict = torch.load(join(CURRENT_DIR, ".." ,"data", "enzyme_data", "all_sequences_esm1b_ts_mean.pt"))

UNIPROT_df["ESM1b_ts_mean"] = ""

for ind in UNIPROT_df.index:
    try:
        UNIPROT_df["ESM1b_ts_mean"][ind] = rep_dict[str(ind) +".pt"]
    except:
        print(ind)
UNIPROT_df

In [ ]:
Uniprot_df = pd.DataFrame(data = {"Uniprot ID" : UNIPROT_df["Uniprot ID"],
                                 "ESM1b_ts_mean" : UNIPROT_df["ESM1b_ts_mean"]})

df_train = df_train.merge(Uniprot_df, on = "Uniprot ID", how = "left")
df_test = df_test.merge(Uniprot_df, on = "Uniprot ID", how = "left")


df_train.to_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train_with_ESM1b_ts_GNN.pkl"))
df_test.to_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test_with_ESM1b_ts_GNN.pkl"))

## 10. Adding ECFP vectors of different dimensions:

In [ ]:
df_train = pd.read_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train_with_ESM1b_ts_GNN.pkl"))
df_test = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test_with_ESM1b_ts_GNN.pkl"))

df_ecfps = pd.read_pickle(join(CURRENT_DIR, ".." ,"data", "substrate_data", "df_ecfps.pkl"))
df_chebi_to_inchi = pd.read_csv(join(CURRENT_DIR, ".." ,"data", "substrate_data", "chebiID_to_inchi.tsv"), sep = "\t")

In [ ]:
df_ecfps["ECFP_512"] = ""
df_ecfps["ECFP_2048"] = ""

for ind in df_ecfps.index:
    met_ID = df_ecfps["substrate ID"][ind]
    is_CHEBI_ID = (met_ID[0:5] == "CHEBI")
    
    
    if is_CHEBI_ID:
        try:
            ID = int(met_ID.split(" ")[0].split(":")[-1])
            Inchi = list(df_chebi_to_inchi["Inchi"].loc[df_chebi_to_inchi["ChEBI"] == float(ID)])[0]
            if not pd.isnull(Inchi):
                mol = Chem.inchi.MolFromInchi(Inchi)
        except IndexError:
            mol = None
        
    else:
        try:
            mol = Chem.MolFromMolFile(mol_folder +  "/mol-files/" + met_ID + '.mol')
        except OSError:
            None
            
    if mol is not None:
        ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=512).ToBitString()
        df_ecfps["ECFP_512"][ind] = ecfp
        
        ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=2048).ToBitString()
        df_ecfps["ECFP_2048"][ind] = ecfp

In [ ]:
df_train.drop(columns = ["ECFP"], inplace = True)
df_train = df_train.merge(df_ecfps, on = "substrate ID", how = "left")

df_test.drop(columns = ["ECFP"], inplace = True)
df_test = df_test.merge(df_ecfps, on = "substrate ID", how = "left")

In [ ]:
df_train.to_pickle(join(CURRENT_DIR, ".." ,"data","splits", "df_train_with_ESM1b_ts_GNN.pkl"))
df_test.to_pickle(join(CURRENT_DIR, ".." ,"data", "splits", "df_test_with_ESM1b_ts_GNN.pkl"))